In [ ]:
 (async function() {
     // 1. استخراج matchId تلقائياً من رابط الصفحة المفتوحة
     let matchId = null;
     const hashMatch = window.location.hash.match(/id:(\d+)/);
     if (hashMatch) {
         matchId = hashMatch[1];
     } else {
         const pathMatch = window.location.pathname.match(/\/(\d+)$/);
         if (pathMatch) matchId = pathMatch[1];
     }

     if (!matchId) {
         console.error("[-] لم يتم العثور على matchId تلقائياً من الرابط! تأكد أنك داخل صفحة الماتش.");
         return;
     }

     console.log(`[+] جاري استخراج بيانات الماتش رقم: ${matchId}...`);
    
     // 2. جلب البيانات من الـ API
     const [resEvent, resLineups] = await Promise.all([
         fetch(`/api/v1/event/${matchId}`).then(r => r.json()),
         fetch(`/api/v1/event/${matchId}/lineups`).then(r => r.json())
     ]);

     const event = resEvent.event || {};
     const homeTeam = event.homeTeam?.name || "Home";
     const awayTeam = event.awayTeam?.name || "Away";
     const homeScore = event.homeScore?.current || 0;
     const awayScore = event.awayScore?.current || 0;

     let rows = [];

     function processPlayers(players, teamName, isHome) {
         (players || []).forEach(p => {
             if (!p.statistics) return;
             const s = p.statistics;
             const minutes = s.minutesPlayed || 0;
            
             // حسابات الالتحامات المجمعة
             const duelWon = s.duelWon || 0;
             const duelLost = s.duelLost || 0;
             const totalDuels = duelWon + duelLost;

             const aerWon = s.aerialWon || s.aerialDuelWon || 0;
             const aerLost = s.aerialLost || s.aerialDuelLost || 0;
             const aerTotal = aerWon + aerLost;

             const grdWon = Math.max(0, duelWon - aerWon);
             const grdTotal = Math.max(0, totalDuels - aerTotal);

             // الكلين شيت
             const conceded = isHome ? awayScore : homeScore;
             const cleanSheet = (conceded === 0 && minutes >= 60 && ['G', 'D'].includes(p.position)) ? 1 : 0;

             // البيانات الأساسية للجدول والعرض
             const baseData = {
                 match_id: Number(matchId),
                 player_id: p.player?.id || null,
                 player_name: p.player?.name || "",
                 team: teamName,
                 position: p.position || "",
                 minutes_played: minutes,
                 sofascore_rating: s.rating || 0.0,
                 clean_sheet: cleanSheet,
                 display_passes: `${s.accuratePass || 0}/${s.totalPass || 0}`,
                 display_total_duels: `${totalDuels} (${duelWon})`,
                 display_ground_duels: `${grdTotal} (${grdWon})`,
                 display_aerial_duels: `${aerTotal} (${aerWon})`,
                 ground_duels_won: grdWon,
                 ground_duels_total: grdTotal,
                 aerial_duels_won: aerWon,
                 aerial_duels_total: aerTotal
             };

             // دمج الإحصائيات الخام وحذف الكائنات المتداخلة غير الرقمية
             const rawStats = { ...s };
             delete rawStats.ratingVersions;
             delete rawStats.statisticsType;

             rows.push({
                 ...baseData,
                 ...rawStats
             });
         });
     }

     processPlayers(resLineups.home?.players, homeTeam, true);
     processPlayers(resLineups.away?.players, awayTeam, false);

     // 3. تنزيل الملف بالـ ID الخاص بالمباراة الحالية تلقائياً
     const blob = new Blob([JSON.stringify(rows, null, 2)], { type: 'application/json' });
     const url = URL.createObjectURL(blob);
     const a = document.createElement('a');
     a.href = url;
     a.download = `match_${matchId}_full_stats.json`;
     document.body.appendChild(a);
     a.click();
     document.body.removeChild(a);
     console.log(`✓ تم بنجاح! تم حفظ ملف match_${matchId}_full_stats.json بعدد ${rows.length} لاعب.`);
 })();

In [3]:
!mamba install pandas

mambajs 0.21.4

Specs: xeus-python, numpy, matplotlib, pillow, ipywidgets>=8.1.6, ipyleaflet, scipy, pandas
Channels: emscripten-forge-4x, conda-forge

Solving environment...
Solving took 0.918 seconds
  Name    Version  Build                Channel
-------------------------------------------------------------
+ pandas  3.1.0    np23py313h124dbc7_0  emscripten-forge-4x
- pip     26.2.2   pyh145f28c_0         conda-forge


In [6]:
import glob
import pandas as pd

# قراءة كل ملفات الـ JSON في الفولدر الحالي
json_files = glob.glob("match_*.json")
if not json_files:
    json_files = glob.glob("match_*")

print(f"[+] جاري دمج {len(json_files)} ملفات شاملة كل اللاعبين...")

dataframes = []
for file in json_files:
    try:
        df_match = pd.read_json(file)
        dataframes.append(df_match)
    except Exception as e:
        print(f"[-] خطأ في {file}: {e}")

if dataframes:
    # 1. دمج جميع اللاعبين بلا أي حذف أو استبعاد
    df_all = pd.concat(dataframes, ignore_index=True)

    # 2. ملء كل الخانات المفقودة في الأعمدة الرقمية بـ 0
    numeric_cols = df_all.select_dtypes(include=['float64', 'int64']).columns
    df_all[numeric_cols] = df_all[numeric_cols].fillna(0)

    # 3. ملء الخانات النصية الفارغة بشرطة
    df_all = df_all.fillna("-")

    # 4. حفظ الملف المجمع الكامل
    output_filename = "game_week_2.csv"
    df_all.to_csv(output_filename, index=False, encoding="utf-8-sig")

    print(f"[✓] تم الحفظ بنجاح بدون استبعاد أي لاعب!")
    print(f"[✓] إجمالي عدد اللاعبين في الجولة (أساسي + دكة): {len(df_all)}")
    print(f"[✓] اسم الملف: {output_filename}")
    display(df_all.head())

[+] جاري دمج 10 ملفات شاملة كل اللاعبين...
[✓] تم الحفظ بنجاح بدون استبعاد أي لاعب!
[✓] إجمالي عدد اللاعبين في الجولة (أساسي + دكة): 400
[✓] اسم الملف: game_week_2.csv


,match_id,player_id,player_name,team,position,minutes_played,sofascore_rating,clean_sheet,display_passes,display_total_duels,...,passValueNormalized,dribbleValueNormalized,defensiveValueNormalized,goalkeeperValueNormalized,bestBallCarryProgression,totalProgressiveBallCarriesDistance,keyPass,expectedGoals,shotValueNormalized,expectedGoalsOnTarget
0,16738930,1094001,Ahmed Mayhoub,Smouha,G,90,8.0,1,16/18,1 (1),...,0.05,0.03,0.23,0.44,0.000000,0.000000,0.0,0.0000,0.00,0.0
1,16738930,1136093,Hesham Hafez,Smouha,D,90,7.6,1,23/33,16 (10),...,0.32,0.04,0.20,0.00,27.782033,78.534744,1.0,0.0000,0.00,0.0
2,16738930,934781,Mohamed Ragab,Smouha,D,90,6.9,1,25/36,11 (5),...,0.16,0.02,0.35,0.00,17.872945,19.794734,0.0,0.0000,0.00,0.0
3,16738930,966961,Mohamed Desouki,Smouha,D,90,7.1,1,26/34,7 (4),...,0.13,0.03,0.27,0.00,0.000000,0.000000,0.0,0.1229,-0.02,0.0
4,16738930,918908,Abdel Rahman Amer,Smouha,D,90,6.9,1,31/43,4 (3),...,0.08,0.09,-0.03,0.00,32.325100,60.085383,1.0,0.0000,0.00,0.0


تاكيدات

In [7]:
print("عدد الماتشات:", df_all['match_id'].nunique())
print("عدد الفرق:", df_all['team'].nunique())
print("قائمة الفرق:\n", df_all['team'].unique())

عدد الماتشات: 10
عدد الفرق: 20
قائمة الفرق:
 <StringArray>
[                'Smouha',        'Asyut Petroleum', 'National Bank of Egypt',
             'Zamalek SC',    'Arab Contractors FC',               'Al-Masry',
        'Modern Sport FC',    'Ghazl El Mahalla FC',    'El-Sharkia Enppi FC',
             'Wadi Degla',            'Pyramids FC', 'Abu Qir Fertilizers SC',
       'Suez Petrojet FC',       'Tala'ea El-Gaish',  'Al-Ittihad Alexandria',
           'Cleopatra FC',               'Canal SC',            'El Gouna FC',
                 'ZED FC',             'Al Ahly FC']
Length: 20, dtype: str


In [8]:
# فحص أقصى وأقل عدد دقائق
print("أقصى دقائق:", df_all['minutes_played'].max())
print("أقل دقائق:", df_all['minutes_played'].min())

# عدد اللاعبين الأساسيين والبدلاء
starters = df_all[df_all['minutes_played'] >= 45]
bench_unplayed = df_all[df_all['minutes_played'] == 0]

print(f"عدد اللي لعبوا 45 دقيقة فأكثر: {len(starters)}")
print(f"عدد اللي قعدوا دكة بدون مشاركة: {len(bench_unplayed)}")

أقصى دقائق: 90
أقل دقائق: 0
عدد اللي لعبوا 45 دقيقة فأكثر: 233
عدد اللي قعدوا دكة بدون مشاركة: 81


In [9]:
# اعرض هدافي الجولة بالكامل للتأكد من الأسماء
top_scorers = df_all[df_all['goals'] > 0][['player_name', 'team', 'goals', 'minutes_played']]
print(top_scorers)

          player_name                   team  goals  minutes_played
72        Chico Banza             Zamalek SC    1.0              45
74      Hossam Ashraf             Zamalek SC    1.0              13
75         Ahmed Esho             Zamalek SC    1.0              10
90       Edrisa Niang    Arab Contractors FC    1.0              82
93         Omar Kamal    Arab Contractors FC    1.0              21
105    Mahmoud Hamada               Al-Masry    1.0              90
111    Hussein Faisal               Al-Masry    1.0              45
112  Mohamed El Shamy               Al-Masry    1.0              45
129   Mahmoud Mamdouh        Modern Sport FC    1.0              87
132     Ahmed Mostafa        Modern Sport FC    1.0              17
170   Mahmoud Kahraba    El-Sharkia Enppi FC    1.0              90
183         Seif Teka             Wadi Degla    1.0              90
190      Hamza Hassan             Wadi Degla    1.0              64
192              Zizo             Wadi Degla    

In [10]:
duplicates = df_all.duplicated(subset=['match_id', 'player_id']).sum()
print("عدد السجلات المكررة:", duplicates) # المفروض يطلع 0

عدد السجلات المكررة: 0


In [11]:
print(df_all['display_passes'].head(30))

0     16/18
1     23/33
2     25/36
3     26/34
4     31/43
5     14/15
6     33/43
7     10/14
8     21/23
9     21/26
10      4/9
11    24/28
12      6/8
13     9/14
14      8/9
15      2/2
16      0/0
17      0/0
18      0/0
19      0/0
20    18/28
21    17/23
22    19/27
23    35/42
24    21/24
25    13/15
26    25/31
27    31/36
28    18/27
29    27/36
Name: display_passes, dtype: str
